# Modelo D — Afinidad supervisada perfil–producto

**Hipótesis:** un algoritmo puede aprender relaciones entre atributos personales y características del producto para ordenar recomendaciones.

**Método:** regresión logística entrenada con reviews positivas y negativas observadas → estimación de probabilidad positiva para cada producto del catálogo → Top-10.

Este es un enfoque nuevo. No se presentan métricas anteriores como si provinieran de este modelo. Tampoco se trata a productos sin review como negativos.

Es un modelo supervisado de clasificación utilizado para construir un ranking; su comparación final utiliza las mismas métricas de recomendación de los otros enfoques.

**Estado:** listo para ejecutar en tu proyecto. Las métricas finales están pendientes; no se incluyen resultados simulados como resultados reales.

## 1. Librerías, datos y split común

Copiá `sephora_common.py` junto a estos notebooks en `notebooks/`. Ese archivo contiene las reglas compartidas de preparación, partición y evaluación. El algoritmo particular de este modelo aparece completo abajo.

Se leen `data/processed/reviews_clean.parquet` y `products_clean.parquet`, con alternativa CSV si no existe el parquet. Ejecutar desde arriba con un kernel que tenga las dependencias instaladas.

In [1]:
from pathlib import Path
import sys
import heapq
from itertools import islice
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# El archivo sephora_common.py debe estar junto a los notebooks.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data" / "processed").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Abrí este notebook dentro del proyecto que contiene data/processed")
helper_dir = next((p for p in [Path.cwd(), PROJECT_ROOT / "notebooks"]
                   if (p / "sephora_common.py").exists()), None)
if helper_dir is None:
    raise FileNotFoundError("Copiá sephora_common.py a notebooks/ junto a estos archivos")
sys.path.insert(0, str(helper_dir))
import sephora_common as sc
from sephora_common import PROFILE_COLS, rank_scores, text_value

# Los mismos valores en los cuatro notebooks.
N_VALIDACION = 500
N_TEST = 500
SEMILLA = 42
ctx = sc.load_context(N_VALIDACION, N_TEST, SEMILLA)
print("Protocolo:", ctx.protocol_id[:12])
print("Resultados:", ctx.output_dir)
display(pd.Series({k:v for k,v in ctx.manifest.items()
                   if k not in ["validation_cases", "test_cases", "versions"]}).to_frame("valor"))
display(pd.DataFrame(ctx.manifest["versions"], index=[0]))

Protocolo: 99fc64b0ea63
Resultados: c:\Users\Administrator\Documents\DATA SCIENCE\PF\sephora-recommender\artifacts\comparacion_4_modelos\99fc64b0ea63


,valor
version,cold-start-users-v2.0
seed,42
protocol_id,99fc64b0ea63f0e1e20f6a8a2d96aee1222e401c635a55...
reviews_input,1093855
reviews_valid_before_pair_dedup,1093855
reviews_unique_pairs,1088886
removed_duplicate_pairs,4969
catalog_skincare,2420
users_total,503216
eligible_users,161396


,python,numpy,pandas,scipy,scikit_learn
0,3.14.5,2.4.6,3.0.3,1.18.1,1.9.0


## 2. Condiciones comunes de comparación

- **Consulta:** tipo y tono de piel, ojos y cabello. Cada enfoque decide qué atributos utiliza. Ninguno recibe compras ni ratings del usuario de consulta.
- **Candidatos:** catálogo Skincare, idéntico para todos. Un modelo basado en evidencia puede puntuar solo parte de este catálogo; ese límite se refleja en la cobertura.
- **Positivo:** `rating >= 4` **o** `is_recommended = True`. La misma regla sirve para todos. Una review de rating bajo pero recomendada es positiva según esta definición.
- **Datos:** un par usuario–producto por vez; se conserva la última review con fecha disponible. Se cuantifica lo descartado. Los perfiles se resumen por moda, sin utilizar ratings para definirlos.
- **Validación:** 500 usuarios para elegir hiperparámetros.
- **Test:** otros 500 usuarios para evaluar la configuración elegida. Se fija un producto positivo por usuario.
- **Sin fuga:** ninguna review de esos 1.000 usuarios entra en entrenamiento, estadísticas de ratings, popularidad o votos.
- **Cold start:** los demás positivos del usuario también permanecen ocultos. No se utilizan ni siquiera para filtrar el Top-10.

Se seleccionan usuarios con al menos dos productos positivos y tipo/tono conocidos. Es una evaluación en esa población, no en todos los visitantes del sitio. Se supone que los atributos personales aportados en las reviews son conocidos al consultar. Esta partición no simula el paso del tiempo.

La huella del protocolo verifica que datos, semilla y tamaños sean iguales. Si cambian, los resultados van a otra carpeta; el comparador no mezcla protocolos.

## 3. Features, objetivo y límites

**Objetivo `y`:** 1 si rating ≥ 4 o recomendación explícita verdadera; 0 en las demás reviews observadas. Por tanto, un 0 no siempre significa un rechazo explícito: incluye valoraciones neutrales según la regla elegida.

**Features:** One-Hot de perfil, marca y categorías; precio transformado con `log1p` y escalado; TF-IDF de categorías, highlights e ingredientes; cruce tipo de piel–categoría y tipo de piel–texto. Estos cruces permiten que un mismo atributo de producto tenga efectos diferentes según el tipo de piel.

**No son features:** rating del catálogo, cantidad de reviews, author_id, product_id ni el resultado de la review. La categoría principal se usa para limitar el catálogo a skincare.

El encoder categórico y la regresión se ajustan con train. TF-IDF y la transformación del precio se ajustan sobre los atributos públicos del catálogo, disponibles para todos los modelos: no utilizan resultados de usuarios de validación/test.

Se usa una muestra de hasta aproximadamente 150.000 reviews de train, estratificada proporcionalmente por clase, para limitar tiempo y memoria. Queda registrado cuántas filas usa. Los otros métodos aprovechan todas las reviews de train: comparar también el presupuesto de datos como una limitación experimental.

`C` controla regularización: cuanto menor, más restringidos los coeficientes. Se ajusta en validación. No usamos umbral 0,5: ordenamos por probabilidad estimada.

**Limitaciones:** aprende de personas que publicaron reviews, no de todas las compras; la probabilidad no está validada como tasa real de satisfacción ni como compatibilidad dermatológica. La clasificación puede favorecer productos populares entre quienes reseñan. El desbalance se conserva y no se fabrican negativos.

In [2]:
class SupervisedAffinity:
    """Regresión logística: perfil + atributos del producto + interacciones.

    No utiliza author_id, product_id, rating del catálogo ni volumen de reviews
    como features. Negativos = reviews no positivas realmente observadas.
    """
    def __init__(self, ctx, max_train_rows=150000, max_text_features=500, seed=42):
        self.ctx = ctx
        self.catalog = ctx.catalog.set_index("product_id", drop=False)
        self.product_order = list(self.catalog.index)
        r = ctx.train.copy()
        self.training_rows_available = len(r)
        if len(r) > max_train_rows:
            # Muestra estratificada aproximadamente proporcional a la clase original.
            portions = []
            for _, group in r.groupby("positive"):
                count = max(1, int(round(max_train_rows*len(group)/len(r))))
                portions.append(group.sample(n=min(count, len(group)), random_state=seed))
            r = pd.concat(portions).sort_values(["author_id", "product_id"])
        self.training_rows_used = len(r)
        self.y = r.positive.astype(int).to_numpy()
        if len(np.unique(self.y)) < 2:
            raise ValueError("La regresión logística necesita reviews positivas y negativas en train")
        self.train_user_ids = set(r.author_id)
        self.types = sorted(set(ctx.profiles.loc[sorted(ctx.train.author_id.unique()), "skin_type"]))
        self.numeric_price = pd.to_numeric(self.catalog.get("price_usd", pd.Series(0, index=self.catalog.index)), errors="coerce")
        self.numeric_price = self.numeric_price.clip(lower=0).fillna(self.numeric_price.median()).fillna(0)
        logprice = np.log1p(self.numeric_price)
        self.price_mean, self.price_std = float(logprice.mean()), float(logprice.std(ddof=0)) or 1.0
        self.price = ((logprice-self.price_mean)/self.price_std).to_dict()
        text_cols = [c for c in ["primary_category", "secondary_category", "tertiary_category", "highlights", "ingredients"] if c in self.catalog]
        texts = self.catalog[text_cols].apply(lambda row: " ".join(text_value(v) for v in row), axis=1)
        self.tfidf = TfidfVectorizer(max_features=max_text_features, stop_words="english", dtype=np.float32)
        try:
            self.X_product_text = self.tfidf.fit_transform(texts)
        except ValueError as e:
            if "empty vocabulary" not in str(e):
                raise
            self.X_product_text = self.tfidf.fit_transform(["skincare"]*len(texts))
        self.product_positions = {pid: i for i, pid in enumerate(self.product_order)}
        profiles = [ctx.profiles.loc[uid].to_dict() for uid in r.author_id]
        pids = r.product_id.tolist()
        self.dict_encoder = DictVectorizer(sparse=True, dtype=np.float32)
        dictionaries = [self.feature_dict(profile, pid) for profile, pid in zip(profiles, pids)]
        self.dict_encoder.fit(dictionaries)
        self.X_train = self.design(profiles, pids)
        self.models = {}
        self.score_cache = {}

    def feature_dict(self, profile, pid):
        product = self.catalog.loc[pid]
        d = {c: str(profile.get(c, "Unknown")) for c in PROFILE_COLS}
        for c in ["brand_name", "secondary_category", "tertiary_category"]:
            d[c] = text_value(product.get(c)) or "Unknown"
        # Cruce explícito: permite aprender afinidades del tipo de piel con categorías.
        d["skin_x_category"] = d["skin_type"]+"|"+d["secondary_category"]
        d["log_price_scaled"] = float(self.price[pid])
        return d

    def design(self, profiles, pids):
        dictionaries = [self.feature_dict(p, pid) for p, pid in zip(profiles, pids)]
        X_cat = self.dict_encoder.transform(dictionaries)
        positions = [self.product_positions[pid] for pid in pids]
        X_text = self.X_product_text[positions]
        blocks = [X_cat, X_text]
        # Interacción tipo de piel × texto: términos pueden tener efectos diferentes.
        skin = np.array([p.get("skin_type", "Unknown") for p in profiles], dtype=object)
        for value in self.types:
            blocks.append(X_text.multiply((skin == value).astype(np.float32)[:, None]))
        return sparse.hstack(blocks, format="csr", dtype=np.float32)

    def fit_config(self, config):
        C = float(config["C"])
        if C not in self.models:
            model = LogisticRegression(C=C, solver="liblinear", max_iter=500, random_state=42)
            model.fit(self.X_train, self.y)
            self.models[C] = model
        return self.models[C]

    def scores(self, profile, config):
        C = float(config["C"])
        key = (tuple(profile.get(c, "Unknown") for c in PROFILE_COLS), C)
        if key not in self.score_cache:
            model = self.fit_config(config)
            X = self.design([profile]*len(self.product_order), self.product_order)
            probabilities = model.predict_proba(X)[:, 1]
            self.score_cache[key] = pd.DataFrame({"product_id": self.product_order, "score": probabilities})
        return self.score_cache[key]

    def recommend(self, profile, config, top_n=10, exclude=()):
        return rank_scores(self.scores(profile, config), "score", top_n, exclude)

## 4. Preparación del modelo y configuraciones

In [3]:
MAX_TRAIN_ROWS = 150000
MAX_TEXT_FEATURES = 500
print("Preparando features del modelo supervisado; puede tardar...")
model = SupervisedAffinity(ctx, max_train_rows=MAX_TRAIN_ROWS,
                           max_text_features=MAX_TEXT_FEATURES, seed=SEMILLA)
CONFIGURACIONES = [{"C": c} for c in [0.1, 1.0, 10.0]]
print("Reviews disponibles:", model.training_rows_available)
print("Reviews usadas:", model.training_rows_used)
print("Matriz de entrenamiento:", model.X_train.shape)
print("Proporción positiva en train:", model.y.mean())

Preparando features del modelo supervisado; puede tardar...
Reviews disponibles: 1084636
Reviews usadas: 150000
Matriz de entrenamiento: (150000, 3282)
Proporción positiva en train: 0.8454133333333333


## 5. Ajuste en validación y evaluación final

**Hit Rate@10:** usuarios cuyo producto oculto aparece en el Top-10 / usuarios evaluados.

**NDCG@10:** vale 0 si no aparece; si aparece en posición `r`, vale `1/log2(r+1)`. Con un solo positivo de prueba por usuario, premia que el acierto esté más arriba.

**Cobertura de usuarios:** porcentaje que recibe al menos una recomendación. **Top-10 completo:** porcentaje que recibe diez. **Cobertura de catálogo:** productos distintos recomendados / productos de skincare del catálogo.

Los usuarios sin recomendaciones cuentan como misses. No se eliminan del denominador. El parámetro se elige por HR@10 de validación y, en empate, por NDCG@10. Un empate total conserva la primera configuración de la lista. Test se utiliza después de esa elección.

Se compara con popularidad positiva global calculada exclusivamente sobre train. El promedio de tiempo por consulta es orientativo: puede incluir efectos de caché y no constituye un benchmark de producción.

In [4]:
mejor_config, validacion, metricas_test, detalle_test, baseline_test = sc.run_experiment(
    model, ctx, CONFIGURACIONES)
print("Resultados de validación:")
display(validacion)
print("Configuración elegida:", mejor_config)
print("Resultados finales en test:")
display(pd.DataFrame([metricas_test, baseline_test], index=["Modelo", "Baseline global"]))

Validación: {'C': 0.1}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'C': 1.0}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'C': 10.0}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Configuración seleccionada: {'C': 0.1} 
Evaluación test
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Resultados de validación:


,config,users,hits,hit_rate_at_10,ndcg_at_10,user_coverage,full_top10_rate,catalog_coverage,mean_recommendations,mean_query_seconds
0,{'C': 0.1},500,5,0.010,0.003263,1.0,1.0,0.009917,10.0,0.422993
1,{'C': 1.0},500,1,0.002,0.000631,1.0,1.0,0.011983,10.0,0.401716
2,{'C': 10.0},500,0,0.000,0.000000,1.0,1.0,0.011157,10.0,0.412651


Configuración elegida: {'C': 0.1}
Resultados finales en test:


,users,hits,hit_rate_at_10,ndcg_at_10,user_coverage,full_top10_rate,catalog_coverage,mean_recommendations,mean_query_seconds
Modelo,500,3,0.006,0.002106,1.0,1.0,0.009917,10.0,0.160877
Baseline global,500,37,0.074,0.033161,1.0,1.0,0.004132,10.0,0.005851


## 6. Ejemplo de Top-10 para una persona nueva

Todos los notebooks usan el mismo perfil de ejemplo. Acá no se oculta un producto ni se mide un acierto: se muestra qué devolvería la consulta real. Si no hay evidencia suficiente, la tabla puede tener menos de diez filas.

El ejemplo no reemplaza a la evaluación agregada. No significa que estos productos sean clínicamente adecuados para esa persona.

In [5]:
PERFIL_EJEMPLO = {"skin_type": "oily", "skin_tone": "dark", "eye_color": "blue", "hair_color": "brown"}
perfil, top10 = sc.example_top10(model, ctx, mejor_config, PERFIL_EJEMPLO)
display(pd.DataFrame([perfil]))
display(top10)
print("Productos devueltos:", len(top10))

,skin_type,skin_tone,eye_color,hair_color
0,oily,dark,blue,brown


,position,product_id,product_name,brand_name,secondary_category,price_usd
0,1,P503814,Dynamic Skin Retinol Serum,Dermalogica,Treatments,92.0
1,2,P505031,The POREfessional Smooth Sip Lightweight Gel-C...,Benefit Cosmetics,Moisturizers,44.0
2,3,P505020,The POREfessional Good Cleanup Foaming Cleanser,Benefit Cosmetics,Cleansers,29.0
3,4,P467111,Retinol Acne Clearing Oil,Dermalogica,Moisturizers,86.0
4,5,P505054,The POREfessional Speedy Smooth Mask,Benefit Cosmetics,Masks,39.0
5,6,P442542,Beauty Sleep Power Peel Age-Defying Night Peel,Alpha-H,Treatments,106.0
6,7,P505024,The POREfessional Get Unblocked Makeup-Removin...,Benefit Cosmetics,Cleansers,39.0
7,8,P474822,Daily Glycolic Cleanser,Dermalogica,Cleansers,37.0
8,9,P505009,The POREfessional Tight ’n Toned Pore-Refining...,Benefit Cosmetics,Cleansers,34.0
9,10,P467112,Daily Microfoliant Exfoliator Refill Pack,Dermalogica,Cleansers,56.0


Productos devueltos: 10


## 7. Cierre y exportación de evidencia

La conclusión se calcula con tus resultados nuevos. No reutilizar el 6,2 %, 5,6 %, 2,6 % o 1,4 % de los notebooks anteriores: cambiaron el split y las exclusiones.

Analizar: ¿supera al baseline? ¿Cuánto catálogo cubre? ¿Entrega diez productos? ¿Qué información del perfil usa? ¿Qué sesgos mantiene? Un resultado bajo es evidencia útil, no un motivo para ocultar el experimento.

In [6]:
sc.conclusion(metricas_test, baseline_test)
extra = {"algorithm": "SupervisedAffinity"}
extra.update({"max_train_rows": MAX_TRAIN_ROWS, "rows_used": model.training_rows_used, "rows_available": model.training_rows_available, "text_features": model.X_product_text.shape[1]})

archivo = sc.save_result(ctx, "04_afinidad_supervisada", mejor_config, validacion, metricas_test,
                         detalle_test, baseline_test, top10, extra)
print("Resultado guardado:", archivo)

HR@10 test: 0.60%; baseline: 7.40%; diferencia: -6.80 puntos porcentuales.
Cobertura de usuarios: 100.00%; catálogo: 0.99%.
En esta muestra no supera al baseline en HR. No declararlo ganador por personalización solamente.
Un hit recupera una preferencia observada; no prueba eficacia ni compatibilidad dermatológica.
Resultado guardado: c:\Users\Administrator\Documents\DATA SCIENCE\PF\sephora-recommender\artifacts\comparacion_4_modelos\99fc64b0ea63\04_afinidad_supervisada.json


## 8. Preguntas para entenderlo y defenderlo

1. ¿Qué información usa el algoritmo para ordenar productos?
2. ¿Qué parámetro ajustamos y por qué?
3. ¿Cómo se asegura que el usuario de test no aporte evidencia al ranking?
4. ¿Qué significa un hit y qué no demuestra?
5. ¿Qué aprendimos al compararlo con popularidad global?

Guardá con **Ctrl + S** para conservar todas las salidas y abrí luego `05_comparacion_final.ipynb`.